# Options market return
Run All。Secrets: GITHUB_TOKEN（private repo）、FINLAB_API_TOKEN。正式環境 Python 3.11。

In [ ]:
from pathlib import Path
import os, subprocess, sys
from google.colab import userdata
repo=Path('/content/options_market_return')
try: token=userdata.get('GITHUB_TOKEN')
except Exception: token=None
if not repo.exists():
    import tempfile
    env=os.environ.copy()
    if token:
        env['GITHUB_TOKEN']=token
        with tempfile.TemporaryDirectory() as d:
            ask=Path(d)/'askpass'
            ask.write_text('#!/bin/sh\ncase "$1" in *Username*) echo x-access-token;; *) printf "%s" "$GITHUB_TOKEN";; esac\n')
            ask.chmod(0o700);env['GIT_ASKPASS']=str(ask);env['GIT_TERMINAL_PROMPT']='0'
            subprocess.run(['git','clone','https://github.com/hh4832/options_market_return.git',str(repo)],env=env,check=True)
    else:
        subprocess.run(['git','clone','https://github.com/hh4832/options_market_return.git',str(repo)],check=True)
else:
    remote=subprocess.check_output(['git','remote','get-url','origin'],cwd=repo,text=True).strip()
    if remote!='https://github.com/hh4832/options_market_return.git':raise RuntimeError('Unexpected repository')
    if subprocess.check_output(['git','status','--porcelain'],cwd=repo,text=True).strip():raise RuntimeError('Existing repository is dirty; refusing to update')
    subprocess.run(['git','pull','--ff-only','origin','main'],cwd=repo,check=True)
os.chdir(repo)
print('Git commit:',subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip())
venv_python=repo/'.colab-venv/bin/python'
if sys.version_info[:2]!=(3,11):
    subprocess.run([sys.executable,'-m','pip','install','uv'],check=True)
    if venv_python.exists():
        existing=subprocess.check_output([str(venv_python),'--version'],text=True).strip()
        if not existing.startswith('Python 3.11.'):
            raise RuntimeError(f'Existing .colab-venv is not Python 3.11: {existing}')
    else:
        subprocess.run([sys.executable,'-m','uv','venv','--python','3.11','.colab-venv'],check=True)
    python=str(venv_python)
    subprocess.run([sys.executable,'-m','uv','pip','install','--python',python,'-r','requirements.txt'],check=True)
else:
    python=sys.executable
    subprocess.run([python,'-m','pip','install','-r','requirements.txt'],check=True)
print('Research Python:',subprocess.check_output([python,'--version'],text=True).strip())

In [ ]:
from google.colab import drive
try: finlab_token=userdata.get('FINLAB_API_TOKEN')
except Exception: finlab_token=os.environ.get('FINLAB_API_TOKEN')
if not finlab_token:raise RuntimeError('DATA_VALIDATION_BLOCKED_BY_CREDENTIAL')
os.environ['FINLAB_API_TOKEN']=finlab_token
# Loader calls official finlab.login(token); then validates live datasets.
drive.mount('/content/drive')
DRIVE_OUTPUT_ROOT='/content/drive/MyDrive/00Quant_Research/options_market_return'
if not Path(DRIVE_OUTPUT_ROOT).is_dir():raise RuntimeError('Verify Drive output root')
run_env=os.environ.copy()
subprocess.run([python,'-m','pytest','-q','-p','no:cacheprovider'],check=True,env=run_env)
subprocess.run([python,'-m','src.pipeline'],check=True,env=run_env)

In [ ]:
import pandas as pd
from IPython.display import display, Markdown
runs=sorted(Path('outputs').glob('*/metadata.json'),key=lambda p:p.stat().st_mtime)
if not runs:raise RuntimeError('No run output')
run=runs[-1].parent
print('Output:',run)
display(pd.read_csv(run/'research_results.csv').head(30))
display(Markdown((run/'diagnostics/diagnostics_summary.md').read_text()))
display(pd.read_csv(run/'diagnostics/fdr_diagnostics.csv'))
display(pd.read_csv(run/'observations/observation_summary.csv').head(50))
display(Markdown((run/'observations/observation_summary.md').read_text()))
print('Drive archive:',Path(DRIVE_OUTPUT_ROOT)/run.name)